In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, DotProduct, WhiteKernel, ConstantKernel as C

# Define two training points
X_train = np.array([[0.0], [1.0]])
y_train = np.array([0.0, 1.0])  # So the function is f(0) = 0, f(1) = 1

# Create test points between them
X_test = np.linspace(0, 1, 100).reshape(-1, 1)

# GP with linear kernel (exact DotProduct with sigma_0=1e-5 and fixed noise)
linear_kernel = DotProduct(sigma_0=1e-5) + WhiteKernel(noise_level=1e-5)
gp_linear = GaussianProcessRegressor(kernel=linear_kernel, optimizer=None, alpha=0.0)
gp_linear.fit(X_train, y_train)
y_pred_linear, sigma_linear = gp_linear.predict(X_test, return_std=True)

# GP with RBF kernel
rbf_kernel = RBF(length_scale=0.2) + WhiteKernel(noise_level=1e-5)
gp_rbf = GaussianProcessRegressor(kernel=rbf_kernel, optimizer=None, alpha=0.0)
gp_rbf.fit(X_train, y_train)
y_pred_rbf, sigma_rbf = gp_rbf.predict(X_test, return_std=True)

# Plot results
plt.figure(figsize=(10, 6))
plt.plot(X_test, y_pred_linear, label="Linear Kernel GP", color="blue")
plt.fill_between(X_test.ravel(), y_pred_linear - sigma_linear, y_pred_linear + sigma_linear,
                 alpha=0.2, color="blue")

plt.plot(X_test, y_pred_rbf, label="RBF Kernel GP", color="green")
plt.fill_between(X_test.ravel(), y_pred_rbf - sigma_rbf, y_pred_rbf + sigma_rbf,
                 alpha=0.2, color="green")

plt.plot(X_train, y_train, "ro", label="Train Points")
plt.title("GP Interpolation: Linear vs RBF Kernel (No Optimization)")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.legend()
plt.grid(True)
plt.savefig("plots/gaussian_process_linear_kernel.png", dpi = 300)

In [ ]:
# Define more training points using a nonlinear function
X_train_fun = np.linspace(0, 1, 10).reshape(-1, 1)
y_train_fun = np.sin(2 * np.pi * X_train_fun).ravel()  # Nonlinear function

# Test points
X_test = np.linspace(0, 1, 200).reshape(-1, 1)

# Linear kernel GP
linear_kernel = DotProduct(sigma_0=1e-5) + WhiteKernel(noise_level=1e-5)
gp_linear_fun = GaussianProcessRegressor(kernel=linear_kernel, optimizer=None, alpha=0.0)
gp_linear_fun.fit(X_train_fun, y_train_fun)
y_pred_linear_fun, sigma_linear_fun = gp_linear_fun.predict(X_test, return_std=True)

# Squared Exponential (SE) kernel = RBF kernel
se_kernel = RBF(length_scale=0.2) + WhiteKernel(noise_level=1e-5)
gp_se_fun = GaussianProcessRegressor(kernel=se_kernel, optimizer=None, alpha=0.0)
gp_se_fun.fit(X_train_fun, y_train_fun)
y_pred_se_fun, sigma_se_fun = gp_se_fun.predict(X_test, return_std=True)

# Plotting
plt.figure(figsize=(10, 6))

# Linear GP
plt.plot(X_test, y_pred_linear_fun, label="Linear Kernel GP", color="blue")
plt.fill_between(X_test.ravel(), y_pred_linear_fun - sigma_linear_fun, y_pred_linear_fun + sigma_linear_fun,
                 alpha=0.2, color="blue")

# SE GP
plt.plot(X_test, y_pred_se_fun, label="Squared Exponential Kernel GP", color="green")
plt.fill_between(X_test.ravel(), y_pred_se_fun - sigma_se_fun, y_pred_se_fun + sigma_se_fun,
                 alpha=0.2, color="green")

# Ground truth and training points
plt.plot(X_test, np.sin(2 * np.pi * X_test), "k--", label="True Function")
plt.plot(X_train_fun, y_train_fun, "ro", label="Train Points")

plt.title("GP Regression: Linear vs SE Kernel on Nonlinear Function")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.legend()
plt.grid(True)
plt.savefig("plots/gaussian_process_linear_kernel_multi.png", dpi = 300)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.gaussian_process.kernels import RBF, DotProduct, ExpSineSquared

# Input points from -4 to 4
X = np.linspace(-4, 4, 200).reshape(-1, 1)

# Define kernels
kernels = [
    ("Squared Exponential (RBF)", RBF(length_scale=1.0)),
    ("Linear Kernel", DotProduct(sigma_0=0.0)),
    ("Periodic Kernel", ExpSineSquared(length_scale=1.0, periodicity=1.0))
]

# Create 1x3 subplot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (title, kernel) in zip(axes, kernels):
    K = kernel(X)
    im = ax.imshow(K, extent=[-4, 4, -4, 4], origin="lower", cmap="magma")
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("x'")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Covariance k(x, x')")

plt.tight_layout()
plt.savefig("plots/gaussian_process_kernel_heatmaps.png", dpi = 300)

In [ ]:
import numpy as np
from numpy.linalg import inv

# Squared Exponential kernel
def se_kernel(x1, x2, length_scale=1.0):
    x1 = np.atleast_2d(x1)
    x2 = np.atleast_2d(x2)
    dists = (x1 - x2.T) ** 2
    return np.exp(-0.5 * dists / length_scale**2)

# Setup
X_train = np.array([[0.0], [1.0]])
y_train = np.array([1.0, 3.0])
X_test = np.array([[0.5]])

length_scale = 1.0
epsilon = 1e-5

# Compute kernel matrices
K_xx = se_kernel(X_train, X_train, length_scale)
K_xx_reg = K_xx + epsilon * np.eye(2)
K_xx_inv = inv(K_xx_reg)
K_xs = se_kernel(X_test, X_train, length_scale)

# Posterior mean
mu = K_xs @ K_xx_inv @ y_train

# Display intermediate result: K^-1 @ f
alpha = K_xx_inv @ y_train

# Output everything
{
    "X_train": X_train.ravel().tolist(),
    "X_test": X_test.ravel().tolist(),
    "K(X,X)": K_xx,
    "K(X,X)^-1": K_xx_inv,
    "K(X*,X)": K_xs,
    "alpha = K^-1 f": alpha,
    "mu(X*) = K_* @ alpha": mu.item()
}


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.linalg import inv
from ipywidgets import FloatSlider, Button, VBox, HBox, Output, interactive_output
from matplotlib.gridspec import GridSpec
import os

# Create output widget and global storage
plot_output = Output()
last_fig = {"fig": None, "x_star": 0.5}

# Ensure output directory exists
os.makedirs("plots", exist_ok=True)

# Squared Exponential kernel
def se_kernel(x1, x2, length_scale=1.0):
    x1 = np.atleast_2d(x1)
    x2 = np.atleast_2d(x2)
    dists = (x1 - x2.T) ** 2
    return np.exp(-0.5 * dists / length_scale**2)

# GP and contributions plot
def plot_gp_and_contrib(x_star):
    with plot_output:
        plot_output.clear_output(wait=True)
        
        X_train = np.linspace(0, 1, 10).reshape(-1, 1)
        y_train = np.ones_like(X_train.ravel())
        length_scale = 0.2
        epsilon = 1e-6

        K_xx = se_kernel(X_train, X_train, length_scale)
        K_xx_reg = K_xx + epsilon * np.eye(len(X_train))
        K_xx_inv = inv(K_xx_reg)
        alpha = K_xx_inv @ y_train

        X_test = np.array([[x_star]])
        K_xs = se_kernel(X_test, X_train, length_scale)
        mu = K_xs @ alpha
        contributions = (K_xs * alpha).ravel()

        fig = plt.figure(figsize=(12, 4))
        gs = GridSpec(1, 2, width_ratios=[3, 2])

        ax1 = fig.add_subplot(gs[0])
        ax1.plot(X_train.ravel(), y_train, "bo", label="Training points")
        ax1.axvline(x_star, color="gray", linestyle="--", alpha=0.6, label="X*")
        ax1.plot(x_star, mu, "ro", label=f"μ(X*) = {mu.item():.3f}")
        ax1.set_ylim(-1.5, 1.5)
        ax1.set_xlim(-0.1, 1.1)
        ax1.set_title("GP Posterior Mean")
        ax1.set_xlabel("x")
        ax1.set_ylabel("f(x)")
        ax1.grid(True)
        ax1.legend()

        ax2 = fig.add_subplot(gs[1])
        ax2.bar(range(len(contributions)), contributions, tick_label=[f"{x[0]:.2f}" for x in X_train])
        ax2.set_title("Training Point Contributions")
        ax2.set_xlabel("x_train")
        ax2.set_ylabel("Contribution")
        ax2.set_ylim(-6, 6)
        ax2.grid(True)

        plt.tight_layout()
        plt.show()

        # Save for screenshot
        last_fig["fig"] = fig
        last_fig["x_star"] = x_star

# Screenshot function
def save_screenshot(b):
    fig = last_fig["fig"]
    x_star = last_fig["x_star"]
    if fig is not None:
        filename = f"plots/gaussain_process_interactive_hist_{int(x_star * 100)}.png"
        fig.savefig(filename)
        print(f"Saved screenshot: {filename}")

# UI elements
slider = FloatSlider(min=-0.1, max=1.1, step=0.01, value=0.5, description="x*")
button = Button(description="Screenshot")
button.on_click(save_screenshot)

# Connect manually created slider to plot function
interactive = interactive_output(plot_gp_and_contrib, {'x_star': slider})

# Display final UI
ui = VBox([HBox([slider, button]), plot_output])
display(ui, interactive)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.linalg import inv
from ipywidgets import FloatSlider, Button, VBox, HBox, Output, interactive_output
import os

# Ensure output directory exists
os.makedirs("plots", exist_ok=True)

# Create output and figure state
plot_output = Output()
last_fig = {"fig": None, "x_star": 0.5}

# Squared Exponential kernel
def se_kernel(x1, x2, length_scale=1.0):
    x1 = np.atleast_2d(x1)
    x2 = np.atleast_2d(x2)
    dists = (x1 - x2.T) ** 2
    return np.exp(-0.5 * dists / length_scale**2)

# GP contribution line plot
def plot_gp_contrib_line(x_star):
    with plot_output:
        plot_output.clear_output(wait=True)

        X_train = np.linspace(0, 1, 1000).reshape(-1, 1)
        y_train = np.ones_like(X_train.ravel())
        length_scale = 0.05
        epsilon = 1e-6

        K_xx = se_kernel(X_train, X_train, length_scale)
        K_xx_reg = K_xx + epsilon * np.eye(len(X_train))
        K_xx_inv = inv(K_xx_reg)
        alpha = K_xx_inv @ y_train

        X_test = np.array([[x_star]])
        K_xs = se_kernel(X_test, X_train, length_scale)
        mu = K_xs @ alpha
        contributions = (K_xs * alpha).ravel()

        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(X_train.ravel(), contributions, label="Contribution", color="tab:blue")
        ax.axvline(x_star, color="gray", linestyle="--", alpha=0.6, label="X*")
        ax.axhline(0, color="black", linewidth=0.8, alpha=0.4)
        ax.set_title(f"GP Contribution from 1000 Training Points\nμ(X*) = {mu.item():.3f}")
        ax.set_xlabel("x_train")
        ax.set_ylabel("Contribution to μ(X*)")
        ax.set_xlim(-0.15, 1.15)
        ax.set_ylim(-6, 6)
        ax.grid(True)
        ax.legend()
        plt.tight_layout()
        plt.show()

        # Store for screenshot
        last_fig["fig"] = fig
        last_fig["x_star"] = x_star

# Screenshot saving
def save_screenshot(b):
    fig = last_fig["fig"]
    x_star = last_fig["x_star"]
    if fig is not None:
        filename = f"plots/gaussain_process_lineplot_{int(x_star * 1000)}.png"
        fig.savefig(filename)
        print(f"Saved screenshot: {filename}")

# Slider and button
slider = FloatSlider(min=-0.1, max=1.1, step=0.001, value=0.5, description="x*")
button = Button(description="📸 Save Screenshot")
button.on_click(save_screenshot)

# Link slider to plot
interactive = interactive_output(plot_gp_contrib_line, {'x_star': slider})

# UI layout
ui = VBox([HBox([slider, button]), plot_output])
display(ui, interactive)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.linalg import inv
from ipywidgets import FloatSlider, Button, VBox, HBox, Output, interactive_output
import os

# ── setup ───────────────────────────────────────────────────────────────────────
os.makedirs("plots", exist_ok=True)          # make sure plots/ exists
plot_output = Output()                       # where the figure is shown
last_fig = {"fig": None, "x_star": 0.5}      # keep the last figure for saving

# ── kernel ──────────────────────────────────────────────────────────────────────
def se_kernel(x1, x2, length_scale=1.0):
    x1, x2 = np.atleast_2d(x1), np.atleast_2d(x2)
    dists = (x1 - x2.T) ** 2
    return np.exp(-0.5 * dists / length_scale**2)

# ── main plotting function ─────────────────────────────────────────────────────
def plot_gp_and_contributions(x_star):
    with plot_output:
        plot_output.clear_output(wait=True)

        # training data & GP setup
        X_train = np.linspace(-1.0, 2.0, 10).reshape(-1, 1)
        f_true = lambda x: np.sin(2*np.pi*x) + 0.3*np.cos(6*np.pi*x)
        y_train = f_true(X_train.ravel())
        length_scale, eps = 0.2, 1e-6

        K_xx = se_kernel(X_train, X_train, length_scale)
        K_inv = np.linalg.inv(K_xx + eps * np.eye(len(X_train)))
        alpha = K_inv @ y_train

        # prediction at x_star
        X_test = np.array([[x_star]])
        K_xs = se_kernel(X_test, X_train, length_scale)
        mu_star = K_xs @ alpha
        contributions = (K_xs * alpha).ravel()

        # full posterior curve
        X_pred = np.linspace(-1.0, 2.0, 500).reshape(-1, 1)
        K_pred = se_kernel(X_pred, X_train, length_scale)
        y_pred = K_pred @ alpha
        K_pp = se_kernel(X_pred, X_pred, length_scale)
        var_diag = np.clip(
            np.diag(K_pp) - np.sum(K_pred @ K_inv * K_pred, axis=1), 0, None
        )
        std_pred = np.sqrt(var_diag)

        # ── plotting ────────────────────────────────────────────────────────────
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), height_ratios=[2, 1])

        ax1.plot(X_pred, f_true(X_pred), color="gray", alpha=0.5, label="Ground Truth f(x)")
        ax1.plot(X_pred, y_pred, color="tab:blue", label="GP Posterior Mean")
        ax1.fill_between(
            X_pred.ravel(), y_pred-2*std_pred, y_pred+2*std_pred,
            color="tab:blue", alpha=0.2, label="±2σ"
        )
        ax1.plot(x_star, mu_star, "ro", label=f"μ(X*) = {mu_star.item():.3f}")
        ax1.axvline(x_star, color="gray", linestyle="--", alpha=0.5)
        ax1.set_title("GP Posterior vs True Function")
        ax1.set_xlim(-1.1, 2.1); ax1.set_ylim(-2.5, 2.5)
        ax1.set_ylabel("f(x)"); ax1.grid(True); ax1.legend()

        ax2.plot(X_train, contributions, color="tab:green", label="Contribution")
        ax2.axhline(0, color="black", linewidth=0.8, alpha=0.4)
        ax2.axvline(x_star, color="gray", linestyle="--", alpha=0.5)
        ax2.set_title("Contribution of Training Points to μ(X*)")
        ax2.set_xlabel("x_train"); ax2.set_ylabel("Contribution")
        ax2.set_xlim(-1.1, 2.1); ax2.set_ylim(-2.5, 2.5)
        ax2.grid(True); ax2.legend()

        plt.tight_layout(); plt.show()

        # store figure for the screenshot button
        last_fig["fig"], last_fig["x_star"] = fig, x_star

# ── screenshot handler ─────────────────────────────────────────────────────────
def save_screenshot(_):
    fig, x_star = last_fig["fig"], last_fig["x_star"]
    if fig is not None:
        fname = f"plots/gaussain_process_gpcontrib_{int(x_star*100)}.png"
        fig.savefig(fname)
        print(f"Saved screenshot → {fname}")

# ── widgets & layout ───────────────────────────────────────────────────────────
slider  = FloatSlider(min=-1.0, max=2.0, step=0.001, value=0.5, description="x*")
button  = Button(description="📸 Save Screenshot")
button.on_click(save_screenshot)

interactive = interactive_output(plot_gp_and_contributions, {"x_star": slider})
ui = VBox([HBox([slider, button]), plot_output])

display(ui, interactive)